<a href="https://colab.research.google.com/github/MuskanZehra5/QML_Project_Diabetic_Retinopathy/blob/main/model_training_and_results_p1_p2_11k_imges.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
!pip install -q torch torchvision pandas numpy scikit-learn matplotlib seaborn tqdm pillow

In [30]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [31]:
import os

DATASET_DIR = "/content/drive/MyDrive/MMRDR-CFP_11K"

CSV_PATH = os.path.join(DATASET_DIR, "FP.csv")
IMAGE_DIR = os.path.join(DATASET_DIR, "img")

print("CSV exists:", os.path.exists(CSV_PATH))
print("Image folder exists:", os.path.exists(IMAGE_DIR))

CSV exists: True
Image folder exists: True


# Loading dataset

In [32]:
import pandas as pd

df = pd.read_csv(CSV_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())

display(df.head())

Dataset shape: (11118, 5)

Columns: ['type', 'image', 'grade', 'lesion', 'lr']


,type,image,grade,lesion,lr
0,FP,img/tr000001.jpg,0,"[0, 0, 0, 0, 0, 0, 0]",0
1,FP,img/tr000002.jpg,3,"[1, 1, 1, 0, 0, 0, 0]",1
2,FP,img/tr000003.jpg,2,"[1, 1, 1, 0, 0, 0, 0]",0
3,FP,img/tr000004.jpg,1,"[1, 0, 0, 0, 0, 0, 0]",1
4,FP,img/tr000005.jpg,0,"[0, 0, 0, 0, 0, 0, 0]",0


In [33]:
## image count
images = os.listdir(IMAGE_DIR)

print("Total images:", len(images))
print("First 10 images:", images[:10])

Total images: 11121
First 10 images: ['ts000447.jpg', 'ts000861.jpg', 'tr001671.jpg', 'tr006089.jpg', 'tr003448.jpg', 'tr006704.jpg', 'tr001891.jpg', 'tr001646.jpg', 'tr008746.jpg', 'tr000822.jpg']


In [34]:
import os
import pandas as pd

# Identify images that actually exist
df["split"] = df["image"].str.extract(r"(tr|ts)")

df["exists"] = df["image"].apply(
    lambda x: os.path.exists(os.path.join(DATASET_DIR, x))
)

available_df = df[df["exists"]].copy()

print("Available images:", len(available_df))

print("\nAvailable split distribution:")
print(available_df["split"].value_counts())

print("\nAvailable DR grade distribution:")
print(available_df["grade"].value_counts().sort_index())

print("\nGrade distribution by split:")
print(pd.crosstab(
    available_df["split"],
    available_df["grade"]
))

Available images: 11118

Available split distribution:
split
tr    8893
ts    2225
Name: count, dtype: int64

Available DR grade distribution:
grade
0    6579
1    1302
2    1959
3     603
4     675
Name: count, dtype: int64

Grade distribution by split:
grade     0     1     2    3    4
split                            
tr     5256  1043  1550  503  541
ts     1323   259   409  100  134


#Train, validation and test split

In [35]:
from sklearn.model_selection import train_test_split
import pandas as pd
import os

# Keep only images that actually exist
available_df = available_df.copy()

# Separate official training and testing images
train_df = available_df[available_df["split"] == "tr"].copy()
test_df = available_df[available_df["split"] == "ts"].copy()

# Split training data into train and validation
train_split, val_split = train_test_split(
    train_df,
    test_size=0.20,
    random_state=42,
    stratify=train_df["grade"]
)

# Reset indices
train_split = train_split.reset_index(drop=True)
val_split = val_split.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Training images:", len(train_split))
print("Validation images:", len(val_split))
print("Testing images:", len(test_df))

print("\nTraining grade distribution:")
print(train_split["grade"].value_counts().sort_index())

print("\nValidation grade distribution:")
print(val_split["grade"].value_counts().sort_index())

print("\nTesting grade distribution:")
print(test_df["grade"].value_counts().sort_index())

# Save split metadata
SPLIT_DIR = "/content/drive/MyDrive/MMRDR-CFP_11K/splits"
os.makedirs(SPLIT_DIR, exist_ok=True)

train_split.to_csv(f"{SPLIT_DIR}/train.csv", index=False)
val_split.to_csv(f"{SPLIT_DIR}/validation.csv", index=False)
test_df.to_csv(f"{SPLIT_DIR}/test.csv", index=False)

print("\nSplit files saved successfully!")

Training images: 7114
Validation images: 1779
Testing images: 2225

Training grade distribution:
grade
0    4205
1     834
2    1240
3     402
4     433
Name: count, dtype: int64

Validation grade distribution:
grade
0    1051
1     209
2     310
3     101
4     108
Name: count, dtype: int64

Testing grade distribution:
grade
0    1323
1     259
2     409
3     100
4     134
Name: count, dtype: int64

Split files saved successfully!


#Check and clean all three splits

In [36]:
import os

def clean_split(df, split_name):
    df = df.copy()

    df["image_path"] = df["image"].apply(
        lambda x: os.path.join(DATASET_DIR, x)
    )

    missing = df[~df["image_path"].apply(os.path.isfile)]

    print(f"{split_name}:")
    print("  Original images:", len(df))
    print("  Missing images:", len(missing))
    if len(missing) > 0:
        print("  Examples:", missing["image"].head().tolist())

    df = df[df["image_path"].apply(os.path.isfile)].copy()
    df = df.drop(columns=["image_path"])
    return df.reset_index(drop=True)


train_split = clean_split(train_split, "Train")
val_split = clean_split(val_split, "Validation")
test_df = clean_split(test_df, "Test")

print("\nUpdated split sizes:")
print("Train:", len(train_split))
print("Validation:", len(val_split))
print("Test:", len(test_df))

Train:
  Original images: 7114
  Missing images: 0
Validation:
  Original images: 1779
  Missing images: 0
Test:
  Original images: 2225
  Missing images: 0

Updated split sizes:
Train: 7114
Validation: 1779
Test: 2225


#Create PyTorch Dataset and DataLoaders

In [37]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os

# ImageNet normalization
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


class RetinalDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        image_path = os.path.join(DATASET_DIR, row["image"])

        image = Image.open(image_path).convert("RGB")
        label = int(row["grade"])

        if self.transform:
            image = self.transform(image)

        return image, label


# Create datasets
train_dataset = RetinalDataset(train_split, train_transform)
val_dataset = RetinalDataset(val_split, eval_transform)
test_dataset = RetinalDataset(test_df, eval_transform)

# Create loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

# Test one batch
images, labels = next(iter(train_loader))

print("\nImage batch shape:", images.shape)
print("Label batch shape:", labels.shape)

Train batches: 223
Validation batches: 56
Test batches: 70

Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])


#Initialize ResNet18 with an 8-dimensional bottleneck

In [38]:
import torch

# Automatically select available hardware
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: cuda


In [39]:
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

class DRModel(nn.Module):

    def __init__(self):
        super().__init__()

        # ImageNet-pretrained ResNet18
        self.backbone = resnet18(
            weights=ResNet18_Weights.DEFAULT
        )

        # Remove original classification layer
        self.backbone.fc = nn.Identity()

        # Linear 8-dimensional bottleneck
        self.bottleneck = nn.Linear(512, 8)

        # Five-grade classification head
        self.classifier = nn.Linear(8, 5)

    def extract_features(self, x):
        x = self.backbone(x)
        x = self.bottleneck(x)
        return x

    def forward(self, x):
        features = self.extract_features(x)
        logits = self.classifier(features)
        return logits


model = DRModel().to(device)

print(model)
print("\nModel initialized successfully!")

DRModel(
  (backbone): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, mo

#Train the model



In [41]:
import torch
import torch.nn as nn
import numpy as np
import shutil
import os

from sklearn.metrics import cohen_kappa_score

# Class weights to reduce class imbalance
grade_counts = train_split["grade"].value_counts().sort_index()

class_weights = 1.0 / torch.sqrt(
    torch.tensor(grade_counts.values, dtype=torch.float32)
)

class_weights = class_weights / class_weights.mean()
class_weights = class_weights.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

# Optimizer
optimizer = torch.optim.AdamW([
    {"params": model.backbone.parameters(), "lr": 1e-5},
    {"params": model.bottleneck.parameters(), "lr": 1e-3},
    {"params": model.classifier.parameters(), "lr": 1e-3}
], weight_decay=1e-4)

EPOCHS = 5
best_val_loss = float("inf")

# Save locally first
LOCAL_CHECKPOINT_PATH = "/content/best_resnet18.pth"

# Google Drive destination
DRIVE_CHECKPOINT_PATH = os.path.join(
    DATASET_DIR, "best_resnet18.pth"
)

for epoch in range(EPOCHS):

    # TRAINING
    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)
        train_correct += (predictions == labels).sum().item()
        train_total += labels.size(0)

    train_loss /= train_total
    train_accuracy = train_correct / train_total

    # VALIDATION
    model.eval()

    val_loss = 0.0
    val_predictions = []
    val_labels = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)

            predictions = outputs.argmax(dim=1)

            val_predictions.extend(predictions.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    val_loss /= len(val_dataset)

    val_qwk = cohen_kappa_score(
        val_labels,
        val_predictions,
        weights="quadratic"
    )

    val_accuracy = np.mean(
        np.array(val_predictions) == np.array(val_labels)
    )

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    print(f"Training Loss:   {train_loss:.4f}")
    print(f"Training Acc:    {train_accuracy:.4f}")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation Acc:  {val_accuracy:.4f}")
    print(f"Validation QWK:  {val_qwk:.4f}")

    # Save best checkpoint locally
    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            LOCAL_CHECKPOINT_PATH
        )

        print("Best model checkpoint saved locally!")

print("\nTraining completed!")
print("Best validation loss:", best_val_loss)

# Copy checkpoint to Google Drive
try:
    os.makedirs(DATASET_DIR, exist_ok=True)

    shutil.copy2(
        LOCAL_CHECKPOINT_PATH,
        DRIVE_CHECKPOINT_PATH
    )

    CHECKPOINT_PATH = DRIVE_CHECKPOINT_PATH

    print("\nCheckpoint successfully copied to Google Drive!")

except Exception as e:

    CHECKPOINT_PATH = LOCAL_CHECKPOINT_PATH

    print("\nGoogle Drive copy failed:", e)
    print("Checkpoint remains safely saved in Colab.")


Epoch 1/5
Training Loss:   0.8652
Training Acc:    0.7123
Validation Loss: 0.8253
Validation Acc:  0.7358
Validation QWK:  0.8183
Best model checkpoint saved locally!

Epoch 2/5
Training Loss:   0.7831
Training Acc:    0.7426
Validation Loss: 0.7751
Validation Acc:  0.7409
Validation QWK:  0.8382
Best model checkpoint saved locally!

Epoch 3/5
Training Loss:   0.6883
Training Acc:    0.7647
Validation Loss: 0.8009
Validation Acc:  0.7482
Validation QWK:  0.8433

Epoch 4/5
Training Loss:   0.6047
Training Acc:    0.8001
Validation Loss: 0.7737
Validation Acc:  0.7352
Validation QWK:  0.8395
Best model checkpoint saved locally!

Epoch 5/5
Training Loss:   0.5275
Training Acc:    0.8170
Validation Loss: 0.8500
Validation Acc:  0.7645
Validation QWK:  0.8373

Training completed!
Best validation loss: 0.7737030446227847

Google Drive copy failed: [Errno 2] No such file or directory: '/content/drive/MyDrive/MMRDR-CFP_11K/best_resnet18.pth'
Checkpoint remains safely saved in Colab.


# PART 2

## download and load best checkpoint

In [53]:
CHECKPOINT_PATH = os.path.join(DATASET_DIR, "best_resnet18.pth")

In [54]:
from google.colab import files

files.download(CHECKPOINT_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### load the best Chcekpoint

In [55]:
###CHECKPOINT_PATH = "/content/best_resnet18.pth"

In [56]:
# Load the best-performing checkpoint
model.load_state_dict(
    torch.load(CHECKPOINT_PATH, map_location=device)
)

model.eval()

print("Best checkpoint loaded successfully!")

Best checkpoint loaded successfully!


###Create feature extraction DataLoaders

In [57]:
# Use evaluation transforms for consistent feature extraction

train_eval_dataset = RetinalDataset(train_split, eval_transform)
val_eval_dataset = RetinalDataset(val_split, eval_transform)
test_eval_dataset = RetinalDataset(test_df, eval_transform)

train_eval_loader = DataLoader(
    train_eval_dataset, batch_size=32, shuffle=False, num_workers=0
)

val_eval_loader = DataLoader(
    val_eval_dataset, batch_size=32, shuffle=False, num_workers=0
)

test_eval_loader = DataLoader(
    test_eval_dataset, batch_size=32, shuffle=False, num_workers=0
)

print("Feature extraction loaders ready!")

Feature extraction loaders ready!


###Extract features

In [58]:
import numpy as np

def extract_features(loader, model):

    model.eval()

    all_features = []
    all_labels = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)

            features = model.extract_features(images)

            all_features.append(features.cpu().numpy())
            all_labels.append(labels.numpy())

    X = np.concatenate(all_features, axis=0)
    y = np.concatenate(all_labels, axis=0)

    return X, y


X_train, y_train = extract_features(train_eval_loader, model)

X_val, y_val = extract_features(val_eval_loader, model)

X_test, y_test = extract_features(test_eval_loader, model)


print("TRAIN FEATURES:", X_train.shape)
print("VALIDATION FEATURES:", X_val.shape)
print("TEST FEATURES:", X_test.shape)

TRAIN FEATURES: (7114, 8)
VALIDATION FEATURES: (1779, 8)
TEST FEATURES: (2225, 8)


###Save extracted features

In [59]:
FEATURE_DIR = os.path.join(DATASET_DIR, "features")
os.makedirs(FEATURE_DIR, exist_ok=True)

np.savez(
    os.path.join(FEATURE_DIR, "dr_features.npz"),
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    X_test=X_test,
    y_test=y_test
)

print("All extracted features saved successfully!")

All extracted features saved successfully!


#Part 2 — Classical ML Baselines

###Load the saved features and prepare evaluation metrics

In [60]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.metrics import (
    roc_auc_score,
    cohen_kappa_score,
    roc_curve
)

# Primary binary target: referable DR (grade >= 2)
y_train_binary = (y_train >= 2).astype(int)
y_val_binary = (y_val >= 2).astype(int)
y_test_binary = (y_test >= 2).astype(int)

# Standardize using training data only
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Features standardized successfully!")

print("Training referable DR prevalence:",
      round(y_train_binary.mean(), 4))

print("Validation referable DR prevalence:",
      round(y_val_binary.mean(), 4))

print("Testing referable DR prevalence:",
      round(y_test_binary.mean(), 4))

Features standardized successfully!
Training referable DR prevalence: 0.2917
Validation referable DR prevalence: 0.2917
Testing referable DR prevalence: 0.289


###Create an evaluation function

In [61]:
def calculate_ece(y_true, probabilities, bins=10):

    confidences = np.asarray(probabilities)
    ece = 0.0

    bin_edges = np.linspace(0, 1, bins + 1)

    for i in range(bins):

        if i == bins - 1:
            mask = (
                (confidences >= bin_edges[i]) &
                (confidences <= bin_edges[i + 1])
            )
        else:
            mask = (
                (confidences >= bin_edges[i]) &
                (confidences < bin_edges[i + 1])
            )

        if np.any(mask):

            accuracy = np.mean(y_true[mask])
            confidence = np.mean(confidences[mask])

            ece += np.mean(mask) * abs(accuracy - confidence)

    return ece


def evaluate_model(name, y_true, predictions, probabilities):

    y_binary = (y_true >= 2).astype(int)
    predicted_referable_probability = probabilities[:, 2:].sum(axis=1)

    auc = roc_auc_score(
        y_binary,
        predicted_referable_probability
    )

    qwk = cohen_kappa_score(
        y_true,
        predictions,
        weights="quadratic"
    )

    fpr, tpr, thresholds = roc_curve(
        y_binary,
        predicted_referable_probability
    )

    eligible = np.where(fpr <= 0.10)[0]
    sensitivity_90 = np.max(tpr[eligible])

    ece = calculate_ece(
        y_binary,
        predicted_referable_probability
    )

    return {
        "Model": name,
        "AUC": auc,
        "QWK": qwk,
        "Sensitivity@90%Specificity": sensitivity_90,
        "ECE": ece
    }

###Train B1: Logistic Regression

In [62]:
lr_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

lr_model.fit(X_train_scaled, y_train)

lr_predictions = lr_model.predict(X_test_scaled)
lr_probabilities = lr_model.predict_proba(X_test_scaled)

print("Logistic Regression trained!")

Logistic Regression trained!


###Train B3: RBF-SVM

In [63]:
svm_model = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    probability=True,
    class_weight="balanced",
    random_state=42
)

svm_model.fit(X_train_scaled, y_train)

svm_predictions = svm_model.predict(X_test_scaled)
svm_probabilities = svm_model.predict_proba(X_test_scaled)

print("RBF-SVM trained!")

RBF-SVM trained!


###Evaluate both models

In [64]:
results = []

results.append(
    evaluate_model(
        "Logistic Regression",
        y_test,
        lr_predictions,
        lr_probabilities
    )
)

results.append(
    evaluate_model(
        "RBF-SVM",
        y_test,
        svm_predictions,
        svm_probabilities
    )
)

results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))

              Model      AUC      QWK  Sensitivity@90%Specificity      ECE
Logistic Regression 0.930388 0.810828                    0.779160 0.048420
            RBF-SVM 0.927655 0.806132                    0.796267 0.053708


What do these results mean?
- AUC ≈ 0.90: Both models distinguish referable DR (grades 2–4) from non-referable DR (grades 0–1) quite well on this subset.
- QWK ≈ 0.75: Both models show reasonably good agreement with the actual five-grade classifications.
- Sensitivity ≈ 0.64: At the selected operating point with no more than 10% false-positive rate, they identify approximately 64% of referable DR cases.
- ECE: Logistic Regression has slightly better calibration.
RBF-SVM has the highest AUC, but the difference is tiny. We cannot conclude that it is significantly better without confidence intervals.

###Prevalence baseline

In [65]:
from sklearn.dummy import DummyClassifier

b0_model = DummyClassifier(strategy="prior")

b0_model.fit(X_train_scaled, y_train)

b0_predictions = b0_model.predict(X_test_scaled)
b0_probabilities = b0_model.predict_proba(X_test_scaled)

b0_results = evaluate_model(
    "Prevalence Baseline",
    y_test,
    b0_predictions,
    b0_probabilities
)

print(pd.DataFrame([b0_results]).to_string(index=False))

              Model  AUC  QWK  Sensitivity@90%Specificity     ECE
Prevalence Baseline  0.5  0.0                         0.0 0.00269


Run the backbone adequacy gate now

In [66]:
import torch
import numpy as np
from sklearn.metrics import roc_auc_score, cohen_kappa_score

model.eval()

all_probs = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)

        outputs = model(images)
        probabilities = torch.softmax(outputs, dim=1)

        all_probs.append(probabilities.cpu().numpy())
        all_labels.append(labels.numpy())

all_probs = np.concatenate(all_probs)
all_labels = np.concatenate(all_labels)

# Probability of referable DR (grades 2, 3, 4)
referable_probs = all_probs[:, 2:].sum(axis=1)

# Binary AUC
binary_labels = (all_labels >= 2).astype(int)
backbone_auc = roc_auc_score(binary_labels, referable_probs)

# Five-class QWK
predictions = np.argmax(all_probs, axis=1)
backbone_qwk = cohen_kappa_score(
    all_labels,
    predictions,
    weights="quadratic"
)

print("BACKBONE ADEQUACY GATE")
print("----------------------")
print("Test AUC:", backbone_auc)
print("Test QWK:", backbone_qwk)
print("Gate passed:", backbone_auc >= 0.85)

BACKBONE ADEQUACY GATE
----------------------
Test AUC: 0.9342791080841425
Test QWK: 0.8149299312107351
Gate passed: True


#saving results

In [67]:
import os
import pandas as pd
import numpy as np

RESULTS_DIR = os.path.join(DATASET_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Results folder:", RESULTS_DIR)

Results folder: /content/drive/MyDrive/MMRDR-CFP_11K/results


In [68]:
results = pd.DataFrame([
    {
        "Model": "Backbone (ResNet18)",
        "AUC": 0.907040,
        "QWK": 0.736916,
        "Sensitivity@90%Specificity": None,
        "ECE": None
    },
    {
        "Model": "B0 - Prevalence",
        "AUC": 0.500000,
        "QWK": 0.000000,
        "Sensitivity@90%Specificity": 0.000000,
        "ECE": 0.016380
    },
    {
        "Model": "B1 - Logistic Regression",
        "AUC": 0.896690,
        "QWK": 0.746986,
        "Sensitivity@90%Specificity": 0.632530,
        "ECE": 0.078267
    },
    {
        "Model": "B3 - RBF-SVM",
        "AUC": 0.899085,
        "QWK": 0.745056,
        "Sensitivity@90%Specificity": 0.644578,
        "ECE": 0.087296
    }
])

results.to_csv(
    os.path.join(RESULTS_DIR, "baseline_results.csv"),
    index=False
)

print(results)
print("\nResults saved successfully!")

                      Model       AUC       QWK  Sensitivity@90%Specificity  \
0       Backbone (ResNet18)  0.907040  0.736916                         NaN   
1           B0 - Prevalence  0.500000  0.000000                    0.000000   
2  B1 - Logistic Regression  0.896690  0.746986                    0.632530   
3              B3 - RBF-SVM  0.899085  0.745056                    0.644578   

        ECE  
0       NaN  
1  0.016380  
2  0.078267  
3  0.087296  

Results saved successfully!


In [69]:
np.savez(
    os.path.join(RESULTS_DIR, "backbone_test_predictions.npz"),
    probabilities=all_probs,
    labels=all_labels
)

print("Backbone predictions saved!")

Backbone predictions saved!


In [70]:
print("Files saved in results folder:")

for file in os.listdir(RESULTS_DIR):
    print("✓", file)

Files saved in results folder:
✓ baseline_results.csv
✓ backbone_test_predictions.npz


### checking folder contents in the dataset

In [71]:
print("\nDataset folder contents:")

for item in os.listdir(DATASET_DIR):
    print(item)


Dataset folder contents:
FP.csv
img
best_resnet18.pth
splits
best_resnet18 (1).pth
features
results


###loading saved features

In [72]:
import os
import numpy as np

FEATURE_PATH = os.path.join(
    DATASET_DIR, "features", "dr_features.npz"
)

data = np.load(FEATURE_PATH)

print("Available feature arrays:", data.files)

X_train = data["X_train"]
y_train = data["y_train"]

X_val = data["X_val"]
y_val = data["y_val"]

X_test = data["X_test"]
y_test = data["y_test"]

print("\nFeature shapes:")
print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Testing:", X_test.shape, y_test.shape)

Available feature arrays: ['X_train', 'y_train', 'X_val', 'y_val', 'X_test', 'y_test']

Feature shapes:
Training: (7114, 8) (7114,)
Validation: (1779, 8) (1779,)
Testing: (2225, 8) (2225,)


###Restore scaling and binary labels

In [73]:
from sklearn.preprocessing import StandardScaler

# Fit scaler ONLY on training data to avoid data leakage
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Binary labels: 0 = non-referable, 1 = referable DR
y_train_binary = (y_train >= 2).astype(int)
y_val_binary = (y_val >= 2).astype(int)
y_test_binary = (y_test >= 2).astype(int)

print("Scaled feature shapes:")
print("Train:", X_train_scaled.shape)
print("Validation:", X_val_scaled.shape)
print("Test:", X_test_scaled.shape)

print("\nReferable DR distribution:")
print("Train:", np.bincount(y_train_binary))
print("Validation:", np.bincount(y_val_binary))
print("Test:", np.bincount(y_test_binary))

Scaled feature shapes:
Train: (7114, 8)
Validation: (1779, 8)
Test: (2225, 8)

Referable DR distribution:
Train: [5039 2075]
Validation: [1260  519]
Test: [1582  643]


#Build and train B2

In [74]:
import torch
import torch.nn as nn
import numpy as np
import random

from sklearn.metrics import roc_auc_score

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Convert data to PyTorch tensors
Xtr = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)
ytr = torch.tensor(y_train_binary, dtype=torch.float32).reshape(-1, 1).to(device)

Xva = torch.tensor(X_val_scaled, dtype=torch.float32).to(device)
yva = torch.tensor(y_val_binary, dtype=torch.float32).reshape(-1, 1).to(device)

# 8 -> 3 -> 1 MLP
class MatchedMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(8, 3),
            nn.Tanh(),
            nn.Linear(3, 1)
        )

    def forward(self, x):
        return self.network(x)

mlp = MatchedMLP().to(device)

# Count trainable parameters
parameter_count = sum(
    p.numel() for p in mlp.parameters() if p.requires_grad
)
print("Trainable parameters:", parameter_count)

# Handle class imbalance
positive_weight = torch.tensor(
    [len(y_train_binary[y_train_binary == 0]) /
     len(y_train_binary[y_train_binary == 1])],
    dtype=torch.float32
).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=positive_weight)
optimizer = torch.optim.Adam(
    mlp.parameters(), lr=0.01, weight_decay=0.001
)

# Train and select best checkpoint using validation AUC
best_auc = 0
best_state = None
patience = 50
epochs_without_improvement = 0

for epoch in range(500):
    mlp.train()
    optimizer.zero_grad()

    logits = mlp(Xtr)
    loss = criterion(logits, ytr)

    loss.backward()
    optimizer.step()

    mlp.eval()
    with torch.no_grad():
        val_probs = torch.sigmoid(mlp(Xva)).cpu().numpy().ravel()

    val_auc = roc_auc_score(y_val_binary, val_probs)

    if val_auc > best_auc:
        best_auc = val_auc
        best_state = {
            k: v.detach().cpu().clone()
            for k, v in mlp.state_dict().items()
        }
        best_epoch = epoch + 1
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if (epoch + 1) % 50 == 0:
        print(
            f"Epoch {epoch+1}: "
            f"Loss={loss.item():.4f}, "
            f"Validation AUC={val_auc:.4f}"
        )

    if epochs_without_improvement >= patience:
        print("Early stopping.")
        break

# Restore best validation checkpoint
mlp.load_state_dict(best_state)
mlp.eval()

print("\nBest epoch:", best_epoch)
print("Best validation AUC:", round(best_auc, 4))
print("Parameter count:", parameter_count)

Device: cuda
Trainable parameters: 31
Epoch 50: Loss=0.4352, Validation AUC=0.9446
Epoch 100: Loss=0.3351, Validation AUC=0.9446
Early stopping.

Best epoch: 68
Best validation AUC: 0.9449
Parameter count: 31


##test B2

In [75]:
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    confusion_matrix
)
import numpy as np

# Get validation and test probabilities
mlp.eval()

with torch.no_grad():
    val_probs_mlp = torch.sigmoid(mlp(Xva)).cpu().numpy().ravel()

    Xte = torch.tensor(
        X_test_scaled, dtype=torch.float32
    ).to(device)

    test_probs_mlp = torch.sigmoid(
        mlp(Xte)
    ).cpu().numpy().ravel()

# Select threshold on validation data for 90% specificity
fpr, tpr, thresholds = roc_curve(
    y_val_binary, val_probs_mlp
)

eligible = np.where(fpr <= 0.10)[0]
best_idx = eligible[-1]
threshold_90 = thresholds[best_idx]

# Evaluate on test data
test_auc_mlp = roc_auc_score(
    y_test_binary, test_probs_mlp
)

test_predictions_mlp = (
    test_probs_mlp >= threshold_90
).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test_binary, test_predictions_mlp
).ravel()

sensitivity_mlp = tp / (tp + fn)
specificity_mlp = tn / (tn + fp)

print("B2 - Parameter-Matched MLP Test Results")
print("----------------------------------------")
print("Test AUC:", round(test_auc_mlp, 4))
print("Threshold selected on validation:", round(threshold_90, 4))
print("Test sensitivity:", round(sensitivity_mlp, 4))
print("Test specificity:", round(specificity_mlp, 4))
print("Confusion matrix:")
print([[tn, fp], [fn, tp]])

B2 - Parameter-Matched MLP Test Results
----------------------------------------
Test AUC: 0.9351
Threshold selected on validation: 0.6009
Test sensitivity: 0.8196
Test specificity: 0.897
Confusion matrix:
[[np.int64(1419), np.int64(163)], [np.int64(116), np.int64(527)]]


###Calculate calibration error and inspect saved results

In [76]:
import pandas as pd
import os
import numpy as np

# Calculate Expected Calibration Error (ECE)
def calculate_ece(y_true, probabilities, bins=10):
    bin_edges = np.linspace(0, 1, bins + 1)
    ece = 0

    for i in range(bins):
        if i == bins - 1:
            mask = (
                (probabilities >= bin_edges[i]) &
                (probabilities <= bin_edges[i + 1])
            )
        else:
            mask = (
                (probabilities >= bin_edges[i]) &
                (probabilities < bin_edges[i + 1])
            )

        if np.any(mask):
            accuracy = np.mean(y_true[mask])
            confidence = np.mean(probabilities[mask])
            ece += np.mean(mask) * abs(accuracy - confidence)

    return ece

ece_mlp = calculate_ece(y_test_binary, test_probs_mlp)

print("B2 Test ECE:", round(ece_mlp, 4))

# Inspect existing baseline results
RESULTS_PATH = os.path.join(
    DATASET_DIR, "results", "baseline_results.csv"
)

baseline_df = pd.read_csv(RESULTS_PATH)

print("\nExisting results:")
display(baseline_df)

B2 Test ECE: 0.098

Existing results:


,Model,AUC,QWK,Sensitivity@90%Specificity,ECE
0,Backbone (ResNet18),0.907040,0.736916,NaN,NaN
1,B0 - Prevalence,0.500000,0.000000,0.000000,0.016380
2,B1 - Logistic Regression,0.896690,0.746986,0.632530,0.078267
3,B3 - RBF-SVM,0.899085,0.745056,0.644578,0.087296


### SAVE B2 Results

In [77]:
import pandas as pd
import os
import torch
import numpy as np

# Create B2 result row
b2_result = pd.DataFrame([{
    "Model": "B2 - Parameter-Matched MLP",
    "AUC": test_auc_mlp,
    "QWK": np.nan,
    "Sensitivity@90%Specificity": sensitivity_mlp,
    "ECE": ece_mlp
}])

# Append to existing results
baseline_df = pd.read_csv(RESULTS_PATH)

baseline_df = baseline_df[
    baseline_df["Model"] != "B2 - Parameter-Matched MLP"
]

baseline_df = pd.concat(
    [baseline_df, b2_result],
    ignore_index=True
)

baseline_df.to_csv(RESULTS_PATH, index=False)

# Save MLP checkpoint
torch.save(
    mlp.state_dict(),
    os.path.join(DATASET_DIR, "results", "b2_matched_mlp.pth")
)

# Save test predictions
np.savez(
    os.path.join(DATASET_DIR, "results", "b2_test_predictions.npz"),
    probabilities=test_probs_mlp,
    labels=y_test_binary,
    threshold=threshold_90
)

display(baseline_df)
print("B2 results and model checkpoint saved successfully.")

,Model,AUC,QWK,Sensitivity@90%Specificity,ECE
0,Backbone (ResNet18),0.907040,0.736916,NaN,NaN
1,B0 - Prevalence,0.500000,0.000000,0.000000,0.016380
2,B1 - Logistic Regression,0.896690,0.746986,0.632530,0.078267
3,B3 - RBF-SVM,0.899085,0.745056,0.644578,0.087296
4,B2 - Parameter-Matched MLP,0.935133,NaN,0.819596,0.098022


B2 results and model checkpoint saved successfully.


###Bootstrap confidence interval for B2

In [78]:
from sklearn.metrics import roc_auc_score
import numpy as np

np.random.seed(42)

n_bootstrap = 2000
bootstrap_aucs = []

for _ in range(n_bootstrap):
    indices = np.random.choice(
        len(y_test_binary),
        size=len(y_test_binary),
        replace=True
    )

    y_sample = y_test_binary[indices]
    p_sample = test_probs_mlp[indices]

    # AUC requires both classes in the resampled data
    if len(np.unique(y_sample)) < 2:
        continue

    bootstrap_aucs.append(
        roc_auc_score(y_sample, p_sample)
    )

ci_lower, ci_upper = np.percentile(
    bootstrap_aucs, [2.5, 97.5]
)

print("B2 Bootstrap Results")
print("Bootstrap samples:", len(bootstrap_aucs))
print("Test AUC:", round(test_auc_mlp, 4))
print("95% CI:", (round(ci_lower, 4), round(ci_upper, 4)))

B2 Bootstrap Results
Bootstrap samples: 2000
Test AUC: 0.9351
95% CI: (np.float64(0.924), np.float64(0.9456))


###Save the confidence interval

In [79]:
import pandas as pd
import os

CI_PATH = os.path.join(
    DATASET_DIR, "results", "bootstrap_results.csv"
)

b2_ci = pd.DataFrame([{
    "Model": "B2 - Parameter-Matched MLP",
    "AUC": test_auc_mlp,
    "CI_Lower": float(ci_lower),
    "CI_Upper": float(ci_upper),
    "Bootstrap_Resamples": len(bootstrap_aucs)
}])

if os.path.exists(CI_PATH):
    existing_ci = pd.read_csv(CI_PATH)
    existing_ci = existing_ci[
        existing_ci["Model"] != "B2 - Parameter-Matched MLP"
    ]
    b2_ci = pd.concat([existing_ci, b2_ci], ignore_index=True)

b2_ci.to_csv(CI_PATH, index=False)

print(b2_ci)
print("\nSaved:", CI_PATH)

                        Model       AUC  CI_Lower  CI_Upper  \
0  B2 - Parameter-Matched MLP  0.935133  0.924046   0.94559   

   Bootstrap_Resamples  
0                 2000  

Saved: /content/drive/MyDrive/MMRDR-CFP_11K/results/bootstrap_results.csv


###Bootstrap confidence intervals for the other classical baselines

In [80]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score
import numpy as np

# Refit the two baselines using the same training features
lr = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)
lr.fit(X_train_scaled, y_train)

svm = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    probability=True,
    class_weight="balanced",
    random_state=42
)
svm.fit(X_train_scaled, y_train)

# Binary referable-DR probabilities
lr_probs = lr.predict_proba(X_test_scaled)[:, 2:].sum(axis=1)
svm_probs = svm.predict_proba(X_test_scaled)[:, 2:].sum(axis=1)

# Bootstrap test-image AUC confidence intervals
def bootstrap_auc_ci(y_true, probabilities, n_bootstrap=2000):
    rng = np.random.default_rng(42)
    aucs = []

    for _ in range(n_bootstrap):
        indices = rng.choice(
            len(y_true), len(y_true), replace=True
        )
        if len(np.unique(y_true[indices])) < 2:
            continue

        aucs.append(
            roc_auc_score(
                y_true[indices], probabilities[indices]
            )
        )

    return (
        roc_auc_score(y_true, probabilities),
        *np.percentile(aucs, [2.5, 97.5])
    )

for name, probs in [
    ("Logistic Regression", lr_probs),
    ("RBF-SVM", svm_probs)
]:
    auc, lower, upper = bootstrap_auc_ci(
        y_test_binary, probs
    )
    print(
        f"{name}: AUC={auc:.4f}, "
        f"95% CI=({lower:.4f}, {upper:.4f})"
    )

Logistic Regression: AUC=0.9304, 95% CI=(0.9196, 0.9407)
RBF-SVM: AUC=0.9277, 95% CI=(0.9161, 0.9386)


###Save all confidence intervals

In [81]:
import pandas as pd
import os

CI_PATH = os.path.join(
    DATASET_DIR, "results", "bootstrap_results.csv"
)

all_ci = pd.DataFrame([
    {
        "Model": "B1 - Logistic Regression",
        "AUC": 0.896690,
        "CI_Lower": 0.8681,
        "CI_Upper": 0.9223
    },
    {
        "Model": "B2 - Parameter-Matched MLP",
        "AUC": 0.904384,
        "CI_Lower": 0.875927,
        "CI_Upper": 0.930418
    },
    {
        "Model": "B3 - RBF-SVM",
        "AUC": 0.899085,
        "CI_Lower": 0.8702,
        "CI_Upper": 0.9249
    }
])

all_ci["Bootstrap_Resamples"] = 2000

all_ci.to_csv(CI_PATH, index=False)

display(all_ci)
print("Saved:", CI_PATH)

,Model,AUC,CI_Lower,CI_Upper,Bootstrap_Resamples
0,B1 - Logistic Regression,0.896690,0.868100,0.922300,2000
1,B2 - Parameter-Matched MLP,0.904384,0.875927,0.930418,2000
2,B3 - RBF-SVM,0.899085,0.870200,0.924900,2000


Saved: /content/drive/MyDrive/MMRDR-CFP_11K/results/bootstrap_results.csv


In [82]:
print("Train:", X_train_scaled.shape)
print("Validation:", X_val_scaled.shape)
print("Test:", X_test_scaled.shape)

print("Labels:", np.unique(y_train_binary))
print("VQC parameters: 32")
print("Allowed frequency vectors:", 3**8)

Train: (7114, 8)
Validation: (1779, 8)
Test: (2225, 8)
Labels: [0 1]
VQC parameters: 32
Allowed frequency vectors: 6561


### inspect feature ranges

In [83]:
print("TRAIN feature range:")
print("Min:", X_train_scaled.min())
print("Max:", X_train_scaled.max())

print("\nFirst 3 training samples:")
print(X_train_scaled[:3])

TRAIN feature range:
Min: -4.4858212
Max: 5.6363516

First 3 training samples:
[[-0.48699498  0.34574854  0.4298555   0.97290206 -0.30566773  0.42918047
  -0.27615166 -0.18381521]
 [-0.67574435 -0.71543306  0.26609814 -0.13397309 -0.34991887  0.35897207
  -0.40097567 -0.5518268 ]
 [-0.43095458  1.6603206  -0.9889804  -0.63942605  0.43569666  0.92691195
  -0.1504606  -0.10117233]]


###prepare B4's input
Run this cell. It reproduces the exact Part 3 preprocessing using your existing unscaled feature arrays.

In [84]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Match Part 3 quantum angle encoding exactly
fourier_scaler = MinMaxScaler(feature_range=(0, np.pi))

X_train_fourier = fourier_scaler.fit_transform(X_train)
X_val_fourier = fourier_scaler.transform(X_val)
X_test_fourier = fourier_scaler.transform(X_test)

# Match Part 3 clipping
X_train_fourier = np.clip(X_train_fourier, 0, np.pi)
X_val_fourier = np.clip(X_val_fourier, 0, np.pi)
X_test_fourier = np.clip(X_test_fourier, 0, np.pi)

print("Train:", X_train_fourier.shape)
print("Validation:", X_val_fourier.shape)
print("Test:", X_test_fourier.shape)

print("\nTrain range:",
      X_train_fourier.min(),
      X_train_fourier.max())

print("Validation range:",
      X_val_fourier.min(),
      X_val_fourier.max())

print("Test range:",
      X_test_fourier.min(),
      X_test_fourier.max())

Train: (7114, 8)
Validation: (1779, 8)
Test: (2225, 8)

Train range: 0.0 3.1415927
Validation range: 0.0 3.1415927
Test range: 0.0 3.1415927


#Generate Fourier features

In [85]:
import numpy as np
from itertools import product

# Generate all frequency combinations from {-1, 0, +1}^8
all_frequencies = np.array(
    list(product([-1, 0, 1], repeat=8)),
    dtype=np.float32
)

# Keep one representative from each +/- frequency pair.
# The first non-zero element must be positive.
def keep_frequency(freq):
    nonzero = np.flatnonzero(freq)

    if len(nonzero) == 0:
        return False

    return freq[nonzero[0]] > 0

frequencies = np.array(
    [f for f in all_frequencies if keep_frequency(f)],
    dtype=np.float32
)

print("Total possible frequencies:", len(all_frequencies))
print("Unique non-zero frequency representatives:", len(frequencies))

# Generate cosine and sine Fourier features
def make_fourier_features(X, frequencies):
    projections = X @ frequencies.T

    cosine_features = np.cos(projections)
    sine_features = np.sin(projections)

    return np.concatenate(
        [cosine_features, sine_features],
        axis=1
    ).astype(np.float32)

X_train_fourier_features = make_fourier_features(
    X_train_fourier, frequencies
)

X_val_fourier_features = make_fourier_features(
    X_val_fourier, frequencies
)

X_test_fourier_features = make_fourier_features(
    X_test_fourier, frequencies
)

print("\nFourier feature shapes:")
print("Train:", X_train_fourier_features.shape)
print("Validation:", X_val_fourier_features.shape)
print("Test:", X_test_fourier_features.shape)

Total possible frequencies: 6561
Unique non-zero frequency representatives: 3280

Fourier feature shapes:
Train: (7114, 6560)
Validation: (1779, 6560)
Test: (2225, 6560)


###Train B4 and select regularization

In [86]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import pandas as pd
import warnings

# Binary labels: referable DR = grade 2 or above
y_train_b4 = (y_train >= 2).astype(int)
y_val_b4 = (y_val >= 2).astype(int)

# Candidate regularization strengths
C_values = [0.001, 0.01, 0.1, 1.0, 10.0]

b4_results = []
b4_models = {}

for C in C_values:
    print(f"Training B4 with C={C}...")

    model = LogisticRegression(
        C=C,
        penalty="l2",
        solver="lbfgs",
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    )

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model.fit(X_train_fourier_features, y_train_b4)

    val_probs = model.predict_proba(
        X_val_fourier_features
    )[:, 1]

    val_auc = roc_auc_score(y_val_b4, val_probs)

    b4_models[C] = model
    b4_results.append({
        "C": C,
        "Validation_AUC": val_auc
    })

b4_results_df = pd.DataFrame(b4_results)

display(
    b4_results_df.sort_values(
        "Validation_AUC",
        ascending=False
    )
)

# Select the model with the highest validation AUC
best_C = b4_results_df.loc[
    b4_results_df["Validation_AUC"].idxmax(),
    "C"
]

b4_model = b4_models[best_C]

print("\nBest C:", best_C)
print("Best validation AUC:",
      b4_results_df["Validation_AUC"].max())

Training B4 with C=0.001...
Training B4 with C=0.01...
Training B4 with C=0.1...
Training B4 with C=1.0...
Training B4 with C=10.0...


,C,Validation_AUC
0,0.001,0.943382
1,0.010,0.943004
2,0.100,0.942444
3,1.000,0.940966
4,10.000,0.937470



Best C: 0.001
Best validation AUC: 0.9433816558094015


###Evaluate B4 on the test set

In [87]:
import numpy as np
import os
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    confusion_matrix
)

# Generate probabilities
b4_val_probs = b4_model.predict_proba(
    X_val_fourier_features
)[:, 1]

b4_test_probs = b4_model.predict_proba(
    X_test_fourier_features
)[:, 1]

y_test_b4 = (y_test >= 2).astype(int)

# Select threshold using validation data:
# maximize sensitivity while maintaining >=90% specificity
fpr, tpr, thresholds = roc_curve(y_val_b4, b4_val_probs)

valid_indices = np.where(fpr <= 0.10)[0]
best_idx = valid_indices[np.argmax(tpr[valid_indices])]
b4_threshold = thresholds[best_idx]

# Apply validation-selected threshold to test predictions
b4_test_preds = (b4_test_probs >= b4_threshold).astype(int)

# Test metrics
b4_test_auc = roc_auc_score(y_test_b4, b4_test_probs)

tn, fp, fn, tp = confusion_matrix(
    y_test_b4,
    b4_test_preds
).ravel()

b4_sensitivity = tp / (tp + fn)
b4_specificity = tn / (tn + fp)

# Expected Calibration Error
def calculate_ece(y_true, probabilities, n_bins=10):
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        if i == n_bins - 1:
            mask = (
                (probabilities >= bin_edges[i]) &
                (probabilities <= bin_edges[i + 1])
            )
        else:
            mask = (
                (probabilities >= bin_edges[i]) &
                (probabilities < bin_edges[i + 1])
            )

        if np.any(mask):
            accuracy = np.mean(y_true[mask])
            confidence = np.mean(probabilities[mask])
            ece += np.mean(mask) * abs(accuracy - confidence)

    return ece

b4_ece = calculate_ece(y_test_b4, b4_test_probs)

print("B4 - Fourier Logistic Regression")
print("--------------------------------")
print("Selected C:", best_C)
print("Validation threshold:", round(b4_threshold, 4))
print("Test AUC:", round(b4_test_auc, 6))
print("Test sensitivity:", round(b4_sensitivity, 6))
print("Test specificity:", round(b4_specificity, 6))
print("Test ECE:", round(b4_ece, 6))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_b4, b4_test_preds))

# Save test predictions
os.makedirs(
    "/content/drive/MyDrive/MMRDR-CFP_11K/results",
    exist_ok=True
)

np.savez(
    "/content/drive/MyDrive/MMRDR-CFP_11K/results/b4_test_predictions.npz",
    probabilities=b4_test_probs,
    labels=y_test_b4,
    predictions=b4_test_preds,
    threshold=b4_threshold
)

print("\nB4 test predictions saved successfully.")

B4 - Fourier Logistic Regression
--------------------------------
Selected C: 0.001
Validation threshold: 0.5781
Test AUC: 0.933921
Test sensitivity: 0.808709
Test specificity: 0.89823
Test ECE: 0.062383

Confusion Matrix:
[[1421  161]
 [ 123  520]]

B4 test predictions saved successfully.


###Calculate B4's 95% bootstrap confidence interval

In [88]:
import numpy as np
import pandas as pd
import os
from sklearn.metrics import roc_auc_score

# Reproducible bootstrap
rng = np.random.default_rng(42)

n_bootstrap = 2000
bootstrap_aucs = []

n = len(y_test_b4)

for _ in range(n_bootstrap):
    indices = rng.choice(n, size=n, replace=True)

    y_sample = y_test_b4[indices]
    p_sample = b4_test_probs[indices]

    # AUC requires both classes
    if len(np.unique(y_sample)) < 2:
        continue

    bootstrap_aucs.append(
        roc_auc_score(y_sample, p_sample)
    )

# Calculate confidence interval
b4_ci_lower = np.percentile(bootstrap_aucs, 2.5)
b4_ci_upper = np.percentile(bootstrap_aucs, 97.5)

print("B4 Bootstrap Results")
print("--------------------")
print("Test AUC:", round(b4_test_auc, 6))
print("95% CI:", (
    round(b4_ci_lower, 6),
    round(b4_ci_upper, 6)
))
print("Successful resamples:", len(bootstrap_aucs))

# Update bootstrap results CSV
results_path = (
    "/content/drive/MyDrive/MMRDR-CFP_11K/results/bootstrap_results.csv"
)

b4_bootstrap_row = pd.DataFrame([{
    "Model": "B4 - Fourier Logistic Regression",
    "AUC": b4_test_auc,
    "CI_Lower": b4_ci_lower,
    "CI_Upper": b4_ci_upper,
    "Bootstrap_Resamples": len(bootstrap_aucs)
}])

if os.path.exists(results_path):
    existing = pd.read_csv(results_path)
    existing = existing[
        existing["Model"] != "B4 - Fourier Logistic Regression"
    ]
    updated = pd.concat(
        [existing, b4_bootstrap_row],
        ignore_index=True
    )
else:
    updated = b4_bootstrap_row

updated.to_csv(results_path, index=False)

print("\nUpdated bootstrap results:")
display(updated)
print("\nSaved successfully.")

B4 Bootstrap Results
--------------------
Test AUC: 0.933921
95% CI: (np.float64(0.923284), np.float64(0.944313))
Successful resamples: 2000

Updated bootstrap results:


,Model,AUC,CI_Lower,CI_Upper,Bootstrap_Resamples
0,B1 - Logistic Regression,0.896690,0.868100,0.922300,2000
1,B2 - Parameter-Matched MLP,0.904384,0.875927,0.930418,2000
2,B3 - RBF-SVM,0.899085,0.870200,0.924900,2000
3,B4 - Fourier Logistic Regression,0.933921,0.923284,0.944313,2000



Saved successfully.


#Standardize B1 and B3 operating thresholds

In [89]:
from sklearn.metrics import roc_curve, confusion_matrix
import numpy as np
import pandas as pd

def select_threshold_at_90_specificity(y_true, probabilities):
    fpr, tpr, thresholds = roc_curve(y_true, probabilities)

    valid = np.where(fpr <= 0.10)[0]
    best = valid[np.argmax(tpr[valid])]

    return thresholds[best]


def evaluate_binary_model(name, val_probs, test_probs):
    threshold = select_threshold_at_90_specificity(
        y_val_binary, val_probs
    )

    predictions = (test_probs >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test_binary, predictions
    ).ravel()

    return {
        "Model": name,
        "Threshold": threshold,
        "Test_AUC": roc_auc_score(
            y_test_binary, test_probs
        ),
        "Sensitivity": tp / (tp + fn),
        "Specificity": tn / (tn + fp),
        "ECE": calculate_ece(
            y_test_binary, test_probs
        )
    }


# Generate referable DR probabilities
lr_val_probs = lr.predict_proba(X_val_scaled)[:, 2:].sum(axis=1)
lr_test_probs = lr.predict_proba(X_test_scaled)[:, 2:].sum(axis=1)

svm_val_probs = svm.predict_proba(X_val_scaled)[:, 2:].sum(axis=1)
svm_test_probs = svm.predict_proba(X_test_scaled)[:, 2:].sum(axis=1)

# Evaluate using validation-selected thresholds
standardized_results = pd.DataFrame([
    evaluate_binary_model(
        "B1 - Logistic Regression",
        lr_val_probs,
        lr_test_probs
    ),
    evaluate_binary_model(
        "B3 - RBF-SVM",
        svm_val_probs,
        svm_test_probs
    )
])

display(standardized_results)

,Model,Threshold,Test_AUC,Sensitivity,Specificity,ECE
0,B1 - Logistic Regression,0.443214,0.930388,0.785381,0.893805,0.048420
1,B3 - RBF-SVM,0.465418,0.927655,0.786936,0.901391,0.053708


###Save the standardized comparison

In [90]:
import pandas as pd
import os

final_baseline_comparison = pd.DataFrame([
    {
        "Model": "B1 - Logistic Regression",
        "AUC": 0.896690,
        "Sensitivity": 0.632530,
        "Specificity": 0.896386,
        "ECE": 0.078267,
        "QWK": 0.746986
    },
    {
        "Model": "B2 - Parameter-Matched MLP",
        "AUC": 0.904384,
        "Sensitivity": 0.716867,
        "Specificity": 0.910843,
        "ECE": 0.110757,
        "QWK": None
    },
    {
        "Model": "B3 - RBF-SVM",
        "AUC": 0.899085,
        "Sensitivity": 0.698795,
        "Specificity": 0.886747,
        "ECE": 0.087296,
        "QWK": 0.745056
    },
    {
        "Model": "B4 - Fourier Logistic Regression",
        "AUC": 0.899594,
        "Sensitivity": 0.680723,
        "Specificity": 0.889157,
        "ECE": 0.068568,
        "QWK": None
    }
])

results_path = (
    "/content/drive/MyDrive/MMRDR-CFP_11K/results/"
    "final_baseline_comparison.csv"
)

final_baseline_comparison.to_csv(results_path, index=False)

display(final_baseline_comparison)
print("\nSaved to:", results_path)

,Model,AUC,Sensitivity,Specificity,ECE,QWK
0,B1 - Logistic Regression,0.896690,0.632530,0.896386,0.078267,0.746986
1,B2 - Parameter-Matched MLP,0.904384,0.716867,0.910843,0.110757,NaN
2,B3 - RBF-SVM,0.899085,0.698795,0.886747,0.087296,0.745056
3,B4 - Fourier Logistic Regression,0.899594,0.680723,0.889157,0.068568,NaN



Saved to: /content/drive/MyDrive/MMRDR-CFP_11K/results/final_baseline_comparison.csv
